# Hybrid GNN-RF Explainer


## Local Explainability of the Blend


In [ ]:
import os
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.explain import Explainer, GNNExplainer

import graph_construction
importlib.reload(graph_construction)
from graph_construction import build_graph
from models import GoldenTransformer
from train_and_evaluate import train_random_forest
from resources import OUTPUT_TABLES_PATH, DATA_PATH, DEVICE, MODELS_PATH, OUTPUT_FIGURES_PATH
from data_preparation import prepare_dataset, get_loss_weights


class HybridBlendExplainer(torch.nn.Module):
    """Differentiable GNNExplainer adapter for the fixed 50/50 GNN-RF blend."""

    def __init__(self, gnn, rf_positive_probabilities, gnn_weight=0.5):
        super().__init__()
        self.gnn = gnn
        self.gnn_weight = gnn_weight
        self.register_buffer(
            "rf_positive_probabilities",
            torch.as_tensor(rf_positive_probabilities, dtype=torch.float32),
        )

    def forward(self, x, edge_index, edge_attr=None):
        gnn_probabilities = torch.softmax(
            self.gnn(x, edge_index, edge_attr), dim=-1
        )
        rf_probability = self.rf_positive_probabilities.to(
            device=gnn_probabilities.device,
            dtype=gnn_probabilities.dtype,
        )
        hybrid_positive_probability = (
            self.gnn_weight * gnn_probabilities[:, 1]
            + (1.0 - self.gnn_weight) * rf_probability
        ).clamp(min=1e-7, max=1.0 - 1e-7)
        return torch.stack(
            [
                torch.log1p(-hybrid_positive_probability),
                torch.log(hybrid_positive_probability),
            ],
            dim=-1,
        )


best_transformer_df = pd.read_csv(
    os.path.join(OUTPUT_TABLES_PATH, "best_transformer_df.csv")
)
x, y, pos_combined, pos_spatial, pos_temporal, group_ids = prepare_dataset(
    DATA_PATH, return_group_ids=True
)
class_weights = get_loss_weights(y=y)

log_area_t = x if x.dim() == 2 else x.unsqueeze(1)
x_full = torch.cat([log_area_t, pos_spatial, pos_temporal], dim=1)
rf_features = torch.cat([log_area_t, pos_spatial], dim=1)

best_gnn_row = best_transformer_df.sort_values(by="F1", ascending=False).iloc[0]
best_k, best_h = int(best_gnn_row["K"]), int(best_gnn_row["Heads"])
data = build_graph(
    "multirelational",
    best_k,
    pos_spatial,
    pos_temporal,
    pos_combined,
    x_full,
    y,
    group_ids=group_ids,
).to(DEVICE)

model_gnn = GoldenTransformer(
    input_dim=data.x.size(1),
    hidden_dim=16,
    edge_dim=data.edge_attr.size(1),
    num_heads=best_h,
).to(DEVICE)
weights_path = os.path.join(
    MODELS_PATH, "best_full_features_no_seasonal_weights.pth"
)
model_gnn.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model_gnn.eval()

rf_features_np = rf_features.detach().cpu().numpy()
_, _, model_rf = train_random_forest(
    rf_features,
    data.y,
    data.train_mask,
    data.test_mask,
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1,
)
rf_positive_probabilities = model_rf.predict_proba(rf_features_np)[:, 1]
hybrid_explainer_model = HybridBlendExplainer(
    model_gnn,
    rf_positive_probabilities,
    gnn_weight=0.5,
).to(DEVICE)
hybrid_explainer_model.eval()

explainer = Explainer(
    model=hybrid_explainer_model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(
        mode="multiclass_classification",
        task_level="node",
        return_type="raw",
    ),
)

print(
    f"Hybrid explainer ready: blend=50% GNN + 50% RF, "
    f"GNN=(K={best_k}, heads={best_h}), RF inputs=Log-Area/Latitude/Longitude."
)
print(
    "GNNExplainer masks describe the graph/GNN contribution to the blended "
    "prediction; RF feature effects are measured separately by median replacement."
)


In [ ]:
# -------------------------------------------------------------
# 0. Publication-quality style and output setup
# -------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 16,
    "axes.labelsize": 18,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "axes.linewidth": 1.3,
    "figure.dpi": 300,
    "savefig.bbox": "tight",
})
sns.set_theme(style="whitegrid", font="serif")
os.makedirs(OUTPUT_FIGURES_PATH, exist_ok=True)
os.makedirs(OUTPUT_TABLES_PATH, exist_ok=True)

# -------------------------------------------------------------
# 1. Define prediction categories using the hybrid blend
# -------------------------------------------------------------
with torch.no_grad():
    hybrid_log_probabilities = hybrid_explainer_model(
        data.x, data.edge_index, data.edge_attr
    )
    hybrid_predictions = hybrid_log_probabilities.argmax(dim=-1)

evaluation_indices = data.test_mask.nonzero(as_tuple=False).view(-1)
categories = {
    "Correctly Classified Human": evaluation_indices[
        (hybrid_predictions[evaluation_indices] == 1) & (data.y[evaluation_indices] == 1)
    ],
    "Correctly Classified Natural": evaluation_indices[
        (hybrid_predictions[evaluation_indices] == 0) & (data.y[evaluation_indices] == 0)
    ],
    "Misclassified Human": evaluation_indices[
        (hybrid_predictions[evaluation_indices] == 0) & (data.y[evaluation_indices] == 1)
    ],
    "Misclassified Natural": evaluation_indices[
        (hybrid_predictions[evaluation_indices] == 1) & (data.y[evaluation_indices] == 0)
    ],
}

node_feature_names = ["Log-Area", "Latitude", "Longitude", "Days Since Start"]
rf_feature_names = ["Log-Area", "Latitude", "Longitude"]
feat_data = []
rf_feat_data = []
edge_data = []
dist_data = []

SEED = 42
torch.manual_seed(SEED)
MAX_SAMPLES_PER_CATEGORY = 30
rf_train_mask = data.train_mask.detach().cpu().numpy().astype(bool)
rf_feature_medians = np.median(rf_features_np[rf_train_mask], axis=0)

print("Aggregating local explanations for hybrid predictions...")
for category_name, category_indices in categories.items():
    sampled_indices = category_indices
    if len(category_indices) > MAX_SAMPLES_PER_CATEGORY:
        permutation = torch.randperm(
            len(category_indices), device=category_indices.device
        )[:MAX_SAMPLES_PER_CATEGORY]
        sampled_indices = category_indices[permutation]

    print(f" -> Processing {len(sampled_indices)} samples for: {category_name}")

    for node_tensor in sampled_indices:
        node_idx = node_tensor.item()
        explanation = explainer(
            data.x,
            data.edge_index,
            edge_attr=data.edge_attr,
            index=node_idx,
        )

        # GNNExplainer follows the differentiable GNN path of the hybrid.
        node_mask = explanation.node_mask
        if node_mask.dim() == 2:
            relevant_nodes = node_mask.abs().sum(dim=1) > 0
            selected_node_mask = node_mask[relevant_nodes] if relevant_nodes.any() else node_mask
            gnn_feature_importance = selected_node_mask.mean(dim=0).cpu().numpy()
        else:
            gnn_feature_importance = node_mask.squeeze().cpu().numpy()

        for feature_name, importance in zip(node_feature_names, gnn_feature_importance):
            feat_data.append({
                "Category": category_name,
                "Feature": feature_name,
                "GNN Branch Attribution": float(importance),
            })

        # The RF is not differentiable. Measure its local feature effect by
        # replacing one input with its training-set median and re-evaluating.
        rf_row = rf_features_np[node_idx:node_idx + 1].copy()
        rf_base_probability = model_rf.predict_proba(rf_row)[0, 1]
        for feature_idx, feature_name in enumerate(rf_feature_names):
            perturbed_row = rf_row.copy()
            perturbed_row[0, feature_idx] = rf_feature_medians[feature_idx]
            perturbed_probability = model_rf.predict_proba(perturbed_row)[0, 1]
            signed_effect = float(rf_base_probability - perturbed_probability)
            rf_feat_data.append({
                "Category": category_name,
                "Feature": feature_name,
                "Base RF P(Human)": float(rf_base_probability),
                "Perturbed RF P(Human)": float(perturbed_probability),
                "Signed P(Human) Change": signed_effect,
                "Absolute P(Human) Change": abs(signed_effect),
            })

        # Edge masks explain how graph structure changes the GNN contribution
        # to the hybrid prediction, with the RF probability held fixed.
        edge_mask = explanation.edge_mask
        edge_attributes = explanation.edge_attr
        is_temporal_edge = edge_attributes[:, 3] == 1
        in_explained_subgraph = edge_mask.abs() > 0
        temporal_selection = is_temporal_edge & in_explained_subgraph
        spatial_selection = (~is_temporal_edge) & in_explained_subgraph
        temporal_importance = (
            edge_mask[temporal_selection].mean().item() if temporal_selection.any() else 0.0
        )
        spatial_importance = (
            edge_mask[spatial_selection].mean().item() if spatial_selection.any() else 0.0
        )
        edge_data.extend([
            {"Category": category_name, "Edge Type": "Spatial", "Mean Importance": spatial_importance},
            {"Category": category_name, "Edge Type": "Temporal", "Mean Importance": temporal_importance},
        ])

        explanation_edge_index = explanation.edge_index
        neighbor_mask = explanation_edge_index[1] == node_idx
        connected_nodes = explanation_edge_index[0][neighbor_mask]
        if connected_nodes.numel() > 0:
            neighbor_labels = data.y[connected_nodes]
            human_proportion = (neighbor_labels == 1).sum().item() / connected_nodes.numel()
            natural_proportion = (neighbor_labels == 0).sum().item() / connected_nodes.numel()
        else:
            human_proportion, natural_proportion = 0.0, 0.0
        dist_data.extend([
            {"Category": category_name, "Neighbor Class": "Human-Caused", "Proportion": human_proportion},
            {"Category": category_name, "Neighbor Class": "Natural", "Proportion": natural_proportion},
        ])

# Save machine-readable attribution tables.
df_feat = pd.DataFrame(feat_data)
df_rf_feat = pd.DataFrame(rf_feat_data)
df_edge = pd.DataFrame(edge_data)
df_dist = pd.DataFrame(dist_data)
df_feat.to_csv(os.path.join(OUTPUT_TABLES_PATH, "hybrid_gnn_branch_feature_attributions.csv"), index=False)
df_rf_feat.to_csv(os.path.join(OUTPUT_TABLES_PATH, "hybrid_rf_feature_sensitivity.csv"), index=False)
df_edge.to_csv(os.path.join(OUTPUT_TABLES_PATH, "hybrid_edge_type_attributions.csv"), index=False)
df_dist.to_csv(os.path.join(OUTPUT_TABLES_PATH, "hybrid_neighbor_context.csv"), index=False)

category_order = [
    "Correctly Classified Human",
    "Correctly Classified Natural",
    "Misclassified Human",
    "Misclassified Natural",
]

# -------------------------------------------------------------
# 2. GNN/graph branch: node feature attribution
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(
    data=df_feat,
    x="Feature",
    y="GNN Branch Attribution",
    hue="Category",
    hue_order=category_order,
    errorbar="sd",
    palette="viridis",
    capsize=0.05,
    ax=ax,
)
ax.set_title("Hybrid Prediction: GNN-Branch Node Feature Attribution")
ax.set_xlabel("GNN Input Features")
ax.set_ylabel(r"GNNExplainer Mask Value (mean $\pm$ SD)")
ax.legend(title="Hybrid Prediction Category", frameon=True, loc="upper right")
fig.tight_layout()
figure_path = os.path.join(OUTPUT_FIGURES_PATH, "hybrid_gnn_branch_feature_attributions.png")
fig.savefig(figure_path, dpi=300)
print(f"Saved: {figure_path}")
plt.show()
plt.close(fig)

# -------------------------------------------------------------
# 3. RF branch: local feature sensitivity
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_rf_feat,
    x="Feature",
    y="Absolute P(Human) Change",
    hue="Category",
    hue_order=category_order,
    errorbar="sd",
    palette="magma",
    capsize=0.05,
    ax=ax,
)
ax.set_title("Hybrid Prediction: RF-Branch Local Feature Sensitivity")
ax.set_xlabel("RF Input Features")
ax.set_ylabel("Mean Absolute Change in RF P(Human) after Median Replacement")
ax.legend(title="Hybrid Prediction Category", frameon=True)
fig.tight_layout()
figure_path = os.path.join(OUTPUT_FIGURES_PATH, "hybrid_rf_feature_sensitivity.png")
fig.savefig(figure_path, dpi=300)
print(f"Saved: {figure_path}")
plt.show()
plt.close(fig)

# -------------------------------------------------------------
# 4. Graph branch: spatial/temporal edge attribution
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_edge,
    x="Category",
    y="Mean Importance",
    hue="Edge Type",
    order=category_order,
    errorbar="sd",
    palette="muted",
    capsize=0.08,
    ax=ax,
)
ax.set_title("Hybrid Prediction: GNNExplainer Edge Attribution")
ax.set_xlabel("Hybrid Prediction Category")
ax.set_ylabel(r"Edge Mask Value (mean $\pm$ SD)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
ax.legend(title="Graph Edge Type", frameon=True)
fig.tight_layout()
figure_path = os.path.join(OUTPUT_FIGURES_PATH, "hybrid_edge_type_attributions.png")
fig.savefig(figure_path, dpi=300)
print(f"Saved: {figure_path}")
plt.show()
plt.close(fig)

# -------------------------------------------------------------
# 5. Neighbor context for categories assigned by the hybrid
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=df_dist,
    x="Category",
    y="Proportion",
    hue="Neighbor Class",
    order=category_order,
    errorbar="sd",
    palette="Set2",
    capsize=0.08,
    ax=ax,
)
ax.set_title("Neighbor Class Context for Hybrid Prediction Categories")
ax.set_xlabel("Hybrid Prediction Category")
ax.set_ylabel(r"Mean Neighbor-Class Proportion ($\pm$ SD)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
ax.legend(title="Neighbor Class", frameon=True)
fig.tight_layout()
figure_path = os.path.join(OUTPUT_FIGURES_PATH, "hybrid_neighbor_context.png")
fig.savefig(figure_path, dpi=300)
print(f"Saved: {figure_path}")
plt.show()
plt.close(fig)

print("\nHybrid explainability study complete.")
print("Saved tables: hybrid_gnn_branch_feature_attributions.csv, hybrid_rf_feature_sensitivity.csv, hybrid_edge_type_attributions.csv, hybrid_neighbor_context.csv")
